In [1]:
# Keep repository-relative paths valid from notebook subfolders.
from pathlib import Path
import os

os.chdir(next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "notebooks").is_dir() and (root / "requirements.txt").is_file()
))

# Resolve legacy IDX paths stored in existing CSVs without rewriting the data.
def _relocated_idx_path(value):
    text = str(value).replace("\\", "/")
    old_repo = "AI-Builders-Hackhaton-2026-Backend/"
    if old_repo in text:
        text = text.split(old_repo, 1)[1]
    old_raw = "data/idx_financial_statements/"
    if text.startswith(old_raw):
        text = "data/idx_financial/raw/" + text[len(old_raw):]
    return Path(text)

from pathlib import Path
import pandas as pd
import re
import json

In [2]:
INVENTORY_FILE = Path(
    "data/idx_financial/inventory/idx_dataset_inventory.csv"
)

OUTPUT_FILE = Path(
    "data/idx_financial/inventory/idx_financial_file_scan.csv"
)

print("Inventory exists:", INVENTORY_FILE.exists())

Inventory exists: True


In [3]:
inventory_df = pd.read_csv(
    INVENTORY_FILE
)

xlsx_df = inventory_df[
    inventory_df["extension"]
    .str.lower()
    .eq(".xlsx")
].copy()

print("Total inventory files:", len(inventory_df))
print("Total XLSX files:", len(xlsx_df))

display(
    xlsx_df.head()
)

Total inventory files: 114366
Total XLSX files: 16003


,file_path,file_name,extension,size_mb,year,quarter
3,data\idx_financial_statements\Financial_Statem...,ZYRX_2025_Q1_FS.xlsx,.xlsx,0.338,2025,Q1
12,data\idx_financial_statements\Financial_Statem...,ZONE_2025_Q1_FS.xlsx,.xlsx,0.340,2025,Q1
18,data\idx_financial_statements\Financial_Statem...,ZINC_2025_Q1_FS.xlsx,.xlsx,0.339,2025,Q1
24,data\idx_financial_statements\Financial_Statem...,ZATA_2025_Q1_FS.xlsx,.xlsx,0.304,2025,Q1
29,data\idx_financial_statements\Financial_Statem...,YUPI_2025_Q1_FS.xlsx,.xlsx,0.342,2025,Q1


In [4]:
def extract_ticker(file_name):
    if pd.isna(file_name):
        return None

    match = re.match(
        r"^([A-Z0-9]+)_",
        str(file_name).upper()
    )

    if match:
        return match.group(1)

    return None


xlsx_df["ticker"] = (
    xlsx_df["file_name"]
    .apply(extract_ticker)
)

display(
    xlsx_df[
        [
            "ticker",
            "year",
            "quarter",
            "file_name",
            "file_path"
        ]
    ].head(20)
)

,ticker,year,quarter,file_name,file_path
3,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
12,ZONE,2025,Q1,ZONE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
18,ZINC,2025,Q1,ZINC_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
24,ZATA,2025,Q1,ZATA_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
29,YUPI,2025,Q1,YUPI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
37,YULE,2025,Q1,YULE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
43,YPAS,2025,Q1,YPAS_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
49,YOII,2025,Q1,YOII_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
54,YELO,2025,Q1,YELO_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
59,WTON,2025,Q1,WTON_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...


In [5]:
def resolve_file_path(path_text):
    path = _relocated_idx_path(path_text)

    if path.exists():
        return path

    alternative = Path.cwd() / path

    if alternative.exists():
        return alternative

    return None

In [6]:
MIN_VALID_SIZE_BYTES = 10_000


def check_zip_signature(path):
    try:
        with open(path, "rb") as f:
            signature = f.read(2)

        return signature == b"PK"

    except Exception:
        return False

In [7]:
def scan_excel_file(row):

    resolved_path = resolve_file_path(
        row["file_path"]
    )

    result = {
        "ticker": row.get("ticker"),
        "year": row.get("year"),
        "quarter": row.get("quarter"),
        "file_name": row.get("file_name"),
        "file_path": row.get("file_path"),
        "size_mb": row.get("size_mb"),

        "is_valid_size": False,
        "has_zip_signature": False,
        "can_open_excel": False,

        "sheet_count": None,
        "sheet_names": None,

        "scan_status": None,
        "error_type": None,
        "error_message": None
    }

    if resolved_path is None:
        result["scan_status"] = "FILE_NOT_FOUND"
        return result

    try:
        file_size = resolved_path.stat().st_size

        result["is_valid_size"] = (
            file_size >= MIN_VALID_SIZE_BYTES
        )

        if not result["is_valid_size"]:
            result["scan_status"] = "TOO_SMALL"
            return result

        result["has_zip_signature"] = (
            check_zip_signature(
                resolved_path
            )
        )

        if not result["has_zip_signature"]:
            result["scan_status"] = (
                "INVALID_XLSX_SIGNATURE"
            )
            return result

        excel_file = pd.ExcelFile(
            resolved_path,
            engine="openpyxl"
        )

        sheets = excel_file.sheet_names

        result["can_open_excel"] = True
        result["sheet_count"] = len(sheets)

        result["sheet_names"] = json.dumps(
            sheets,
            ensure_ascii=False
        )

        result["scan_status"] = "VALID"

    except Exception as e:
        result["scan_status"] = "READ_ERROR"
        result["error_type"] = type(e).__name__
        result["error_message"] = str(e)

    return result

In [9]:
from tqdm.auto import tqdm
scan_results = []

total_files = len(xlsx_df)

for _, row in tqdm(
    xlsx_df.iterrows(),
    total=total_files,
    desc="Scanning XLSX files",
    unit="file"
):

    result = scan_excel_file(
        row
    )

    scan_results.append(
        result
    )

e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Scanning XLSX files: 100%|██████████| 16003/16003 [2:29:34<00:00,  1.78file/s]  


In [10]:
scan_df = pd.DataFrame(
    scan_results
)

print(
    "Total scanned:",
    len(scan_df)
)

display(
    scan_df.head(20)
)

Total scanned: 16003


,ticker,year,quarter,file_name,file_path,size_mb,is_valid_size,has_zip_signature,can_open_excel,sheet_count,sheet_names,scan_status,error_type,error_message
0,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.338,True,True,True,32.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
1,ZONE,2025,Q1,ZONE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.340,True,True,True,33.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
2,ZINC,2025,Q1,ZINC_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.339,True,True,True,30.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
3,ZATA,2025,Q1,ZATA_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.304,True,True,True,26.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
4,YUPI,2025,Q1,YUPI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.342,True,True,True,33.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
5,YULE,2025,Q1,YULE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.164,True,True,True,16.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""5220000""...",VALID,NaN,NaN
6,YPAS,2025,Q1,YPAS_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.340,True,True,True,33.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
7,YOII,2025,Q1,YOII_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.115,True,True,True,14.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""6220000""...",VALID,NaN,NaN
8,YELO,2025,Q1,YELO_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.316,True,True,True,28.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN
9,WTON,2025,Q1,WTON_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.371,True,True,True,35.0,"[""Context"", ""InlineXBRL"", ""1000000"", ""1210000""...",VALID,NaN,NaN


In [11]:
status_summary = (
    scan_df[
        "scan_status"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

status_summary.columns = [
    "scan_status",
    "count"
]

display(
    status_summary
)

,scan_status,count
0,VALID,15821
1,TOO_SMALL,180
2,READ_ERROR,2


In [12]:
invalid_files_df = (
    scan_df[
        scan_df[
            "scan_status"
        ] != "VALID"
    ]
    .copy()
)

print(
    "Invalid files:",
    len(invalid_files_df)
)

display(
    invalid_files_df.head(100)
)

Invalid files: 182


,ticker,year,quarter,file_name,file_path,size_mb,is_valid_size,has_zip_signature,can_open_excel,sheet_count,sheet_names,scan_status,error_type,error_message
19,WIKA,2025,Q1,WIKA_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.0,False,False,False,NaN,NaN,TOO_SMALL,NaN,NaN
24,WEGE,2025,Q1,WEGE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.0,False,False,False,NaN,NaN,TOO_SMALL,NaN,NaN
30,VISI,2025,Q1,VISI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.0,False,False,False,NaN,NaN,TOO_SMALL,NaN,NaN
33,VICI,2025,Q1,VICI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.0,False,False,False,NaN,NaN,TOO_SMALL,NaN,NaN
43,UNIC,2025,Q1,UNIC_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.0,False,False,False,NaN,NaN,TOO_SMALL,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1768,PKPK,2024,Q2,PKPK_2024_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.0,False,False,False,NaN,NaN,TOO_SMALL,NaN,NaN
1773,PJAA,2024,Q4,PJAA_2024_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.0,False,False,False,NaN,NaN,TOO_SMALL,NaN,NaN
1791,PGEO,2024,Q2,PGEO_2024_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.0,False,False,False,NaN,NaN,TOO_SMALL,NaN,NaN
1797,PGAS,2024,Q4,PGAS_2024_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0.0,False,False,False,NaN,NaN,TOO_SMALL,NaN,NaN


In [13]:
valid_files_df = (
    scan_df[
        scan_df[
            "scan_status"
        ] == "VALID"
    ]
    .copy()
)

print(
    "Valid XLSX files:",
    len(valid_files_df)
)

print(
    "Unique valid tickers:",
    valid_files_df[
        "ticker"
    ].nunique()
)

Valid XLSX files: 15821
Unique valid tickers: 948


In [14]:
coverage_by_period = (
    valid_files_df
    .groupby(
        [
            "year",
            "quarter"
        ]
    )
    .agg(
        valid_files=(
            "file_name",
            "count"
        ),
        unique_tickers=(
            "ticker",
            "nunique"
        )
    )
    .reset_index()
)

display(
    coverage_by_period
)

,year,quarter,valid_files,unique_tickers
0,2020,Q1,659,659
1,2020,Q2,669,669
2,2020,Q3,676,676
3,2020,Q4,686,686
4,2021,Q1,689,688
5,2021,Q2,716,711
6,2021,Q3,697,690
7,2021,Q4,766,735
8,2022,Q1,736,732
9,2022,Q2,773,770


In [16]:
OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

scan_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "Saved to:",
    OUTPUT_FILE
)

print(
    "Rows saved:",
    len(scan_df)
)

Saved to: data\idx_financial_file_scan.csv
Rows saved: 16003
